In [1]:
import pandas as pd 
import cudf
from cuml.ensemble import RandomForestClassifier
from cuml.linear_model import LogisticRegression
from cuml.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.utils import shuffle
from tqdm import tqdm
import cupy as np
from cuml.common.device_selection import set_global_device_type, get_global_device_type

/home/mike239/.local/lib/python3.10/site-packages/cupy/_environment.py:445: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda11x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


In [2]:
try:
    data = pd.read_csv("Churn.csv")
except Exception:
    data = pd.read_csv("/datasets/Churn.csv")

In [3]:
data = data.dropna(subset=["Tenure"]).reset_index(drop=True)

In [4]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data = pd.get_dummies(data, drop_first=True)
data = data.sample(frac=1)
data = cudf.DataFrame(data)

In [5]:
target = data['Exited']
features = data.drop('Exited', axis=1)

In [6]:
features_t, features_valid, target_t, target_valid = train_test_split(
    features, target, test_size=0.25, random_state=12345)

features_train, features_test, target_train, target_test = train_test_split(
    features_t, target_t, test_size=0.25, random_state=12345)

In [7]:
threshold = 0.32
set_global_device_type("gpu")

In [8]:
print(get_global_device_type)

<function get_global_device_type at 0x7f881f7f3640>


In [9]:
best_model_forest = None
best_f1_score = 0
best_precision = 0
best_recall = 0
best_auc_roc = 0
best_n_estimators_rfc = 0 
best_max_depth_rfc = 0

In [10]:
for i in tqdm(range(1, 100)):
    for g in range(3, 30):
        model = RandomForestClassifier(random_state=12345, n_estimators=i, max_depth=g, n_bins=3)
       
        model.fit(features_train, target_train)
           
        probabilities = model.predict_proba(features_valid)
        probabilities_one = probabilities[1]
        prediction_valid = probabilities_one > threshold
        precision = precision_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_valid))
        recall = recall_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_valid))
        f1 = f1_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_valid))
        roc_auc = roc_auc_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_valid))

        if f1 > best_f1_score:
            best_f1_score = f1
            best_precision = precision
            best_recall = recall
            best_model_forest = model
            best_auc_roc = roc_auc
            best_n_estimators_rfc = i
            best_max_depth_rfc = g

  0%|          | 0/99 [00:00<?, ?it/s]/home/mike239/.local/lib/python3.10/site-packages/cuml/internals/api_decorators.py:344: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return func(**kwargs)
/home/mike239/.local/lib/python3.10/site-packages/cuml/internals/api_decorators.py:188: UserWarning: To use pickling first train using float32 data to fit the estimator
  ret = func(*args, **kwargs)
/home/mike239/.local/lib/python3.10/site-packages/cuml/internals/api_decorators.py:344: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return func(**kwargs)
/home/m

In [11]:
print(f"Best F1 = {'%.4f' % (best_f1_score)}\nAUC-ROC = {'%.4f' % (best_auc_roc)}\nPecision = {'%.4f' % (best_precision)}\nRecall = {'%.4f' % (best_recall)}")

Best F1 = 0.6103
AUC-ROC = 0.7527
Pecision = 0.6091
Recall = 0.6116


In [12]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345)
    
    return features_upsampled, target_upsampled

In [22]:
features_upsampled, target_upsampled = upsample(cudf.DataFrame.to_pandas(features_train), cudf.Series.to_pandas(target_train), 4)
features_upsampled, target_upsampled = cudf.DataFrame(features_upsampled), cudf.Series(target_upsampled)

In [26]:

model_upsemled = RandomForestClassifier(random_state=12345, n_estimators=best_n_estimators_rfc, max_depth=best_max_depth_rfc, n_bins = 2**10)
model_upsemled.fit(features_upsampled, target_upsampled)
        
probabilities_upsemled = model_upsemled.predict_proba(features_valid)
probabilities_one_upsemled = probabilities_upsemled[1]
prediction_upsemled_valid = probabilities_one_upsemled > threshold
precision_upsemled = precision_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_upsemled_valid))
recall_upsemled = recall_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_upsemled_valid))
f1_upsemled = f1_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_upsemled_valid))
roc_auc_upsemled = roc_auc_score(cudf.Series.to_pandas(target_valid), cudf.Series.to_pandas(prediction_upsemled_valid))

print(f"Best F1 = {'%.4f' % (f1_upsemled)}\nAUC-ROC = {'%.4f' % (roc_auc_upsemled)}\nPecision = {'%.4f' % (precision_upsemled)}\nRecall = {'%.4f' % (recall_upsemled)}")

Best F1 = 0.5427
AUC-ROC = 0.7555
Pecision = 0.3923
Recall = 0.8802


In [27]:
probabilities_upsemled = model_upsemled.predict_proba(features_test)
probabilities_one_upsemled = probabilities_upsemled[1]
prediction_upsemled_test = probabilities_one_upsemled > 0.38
precision_upsemled = precision_score(cudf.Series.to_pandas(target_test), cudf.Series.to_pandas(prediction_upsemled_test))
recall_upsemled = recall_score(cudf.Series.to_pandas(target_test), cudf.Series.to_pandas(prediction_upsemled_test))
f1_upsemled = f1_score(cudf.Series.to_pandas(target_test), cudf.Series.to_pandas(prediction_upsemled_test))
roc_auc_upsemled = roc_auc_score(cudf.Series.to_pandas(target_test), cudf.Series.to_pandas(prediction_upsemled_test))

print(f"Best F1 = {'%.4f' % (f1_upsemled)}\nAUC-ROC = {'%.4f' % (roc_auc_upsemled)}\nPecision = {'%.4f' % (precision_upsemled)}\nRecall = {'%.4f' % (recall_upsemled)}")

Best F1 = 0.5360
AUC-ROC = 0.7541
Pecision = 0.4036
Recall = 0.7976
